# Post-training XLS-R deepfake detector (CAC) — v3: robustness fine-tune

This notebook **continues training your existing detector** (the v2 notebook's `best.pt`, the checkpoint the app deploys). It doesn't start over. It reuses the v2 notebook's preprocessed cache on Drive.

**Why a second round.** The 2026-10-22 benchmark of the deployed model found three problems that threshold tuning can't fix:

| Finding | Evidence (per-window logits) | Likely cause in v2 training |
|---|---|---|
| Noise makes AI look human | AI clips 25–27 clean → 12–15 at 10 dB SNR; human 13–16 clean | Real class is M-AILABS audiobooks only (a few speakers, studio noise floor); TTS is clean |
| Digital silence makes humans look AI | A window of 2 s exact zeros + 2 s human speech: 31.3 / 33.8 | Only leading/trailing silence was trimmed; TTS pauses are often exact zeros, audiobook pauses are not |
| Non-speech scores as extreme AI | DTMF ≈ 52, synthetic chord ≈ 48 | Never seen in training |

It also confirmed that the v2 thresholds were picked from float32 `sigmoid` outputs, which round to 1.0 above about logit 16.6.

**What changes (everything is applied identically to real and fake clips, so it can't become a shortcut):**
1. **More real speakers.** LibriSpeech `dev-clean`, `dev-other` and `test-other` (about 100 speakers), split by speaker. `test-clean` is **never** downloaded, because it holds the benchmark's human controls (speakers 1320 and 6930).
2. **Silence augmentation.** Inserted gaps of 0.1–2.5 s (exact zeros or hiss, including at the window edges), noise-floor hiss, and noise-gating that zeroes quiet frames.
3. **Noise and backgrounds.** Noise at 0–30 dB SNR (white, pink, brown, and recorded ESC-50 sounds); music, DTMF and babble overlays; MP3 and Opus codecs; clipping.
4. **A few non-speech windows labeled "not AI speech"** (`NONSPEECH_AS_REAL_P`, 3%) so a gate miss doesn't produce logit 50.
5. **Robust model selection.** The best epoch is the one with the lowest **pooled** validation EER across clean, phone, noise, silence-gap and music conditions, using one threshold for all of them, which is how the live system works.
6. **Thresholds in float64 logit space** (`logit_threshold_eer`, `logit_threshold_1pct_fpr`), plus before/after robustness and non-speech reports.

**How to run (Colab Pro)**
1. The v2 cache must exist at `MyDrive/CAC_deepfake_detector/cache_en/` (it does if the v2 notebook ran), along with the v2 checkpoint at `ckpt_en/best.pt`. If your deployed `best.pt` lives elsewhere, set `INIT_CHECKPOINT`.
2. `Runtime → Change runtime type → L4` (or A100). High-RAM is not needed.
3. `Runtime → Run all`, then approve Google Drive. Downloads total about 1.6 GB (LibriSpeech about 1 GB, ESC-50 about 0.6 GB). Expect roughly **1–1.5 h on an L4** (10–15 min data prep, 3 epochs of about 10–15 min, plus evaluation). This is an estimate from the v2 run's 8 min/epoch; the extra augmentation and data make each epoch longer.
4. If the runtime disconnects, **Run all** again: caches and checkpoints are on Drive and training resumes.

**Optional own data (strongly recommended):** put consented recordings in `MyDrive/CAC_deepfake_detector/own_data/real/<speaker>/` and AI clips in `own_data/fake/<voice>/` (wav, flac, mp3 or m4a). They're used **only for evaluation**, never for training. This is the honest, out-of-corpus check. **Don't** put the three benchmark ElevenLabs clips there if you want to keep the repo benchmark independent.

**Licenses:** MLAAD is CC BY-NC 4.0, LibriSpeech is CC BY 4.0, ESC-50 is CC BY-NC. Fine for a CAC entry; not for a paid product without checking.

## 1. Configuration

In [ ]:
CFG = dict(
    # ---- start from the existing detector ----
    BASE_MODEL       = "facebook/wav2vec2-xls-r-300m",
    KEEP_LAYERS      = 12,
    INIT_CHECKPOINT  = "/content/drive/MyDrive/CAC_deepfake_detector/ckpt_en/best.pt",   # v2 best (dict with "model") or a bare state dict
    EXPECTED_INIT_SHA256 = "ae78eb6ed8e343dedd72351d942a27718eced82f355a90ef7583334a203beec0",  # deployed best.pt; mismatch only warns
    # ---- audio (must match the v2 run and the live agent) ----
    SR = 16000, TRAIN_SEC = 4.0, MAX_STORE_SEC = 8.0, MIN_SEC = 1.0,
    # ---- fine-tuning (starting points, not tuned values) ----
    EPOCHS = 3, PATIENCE = 1,
    LR_BACKBONE = 5e-6, LR_HEAD = 2e-4, WEIGHT_DECAY = 1e-4, WARMUP_FRAC = 0.05, GRAD_CLIP = 1.0,
    # ---- extra real speech (LibriSpeech, CC BY 4.0). NEVER add "test-clean": it holds the benchmark controls ----
    LIBRISPEECH_SUBSETS  = ["dev-clean", "dev-other", "test-other"],   # add "train-clean-100" (6.3 GB, 251 speakers) if time allows
    LIBRISPEECH_MIRRORS  = ["https://openslr.trmal.net/resources/12", "https://openslr.elda.org/resources/12"],
    MAX_UTTS_PER_SPEAKER = 60,
    LIBRISPEECH_LOCAL_DIR = None,   # folder that already contains LibriSpeech/<subset>/... (skips the download)
    # ---- recorded background sounds (ESC-50, CC BY-NC) ----
    USE_ESC50 = True, ESC50_URL = "https://github.com/karoldvl/ESC-50/archive/master.zip", ESC50_MAX_CLIPS = 800,
    ESC50_LOCAL_DIR = None,         # folder containing ESC-50-master/audio (skips the download)
    # ---- augmentation: the train-time function never sees the label ----
    PHONE_AUG_P = 0.7, OPUS_P = 0.12, MP3_P = 0.10,
    NOISE_P = 0.6, NOISE_SNR_DB = (0.0, 30.0), RECORDED_NOISE_FRAC = 0.4,
    OVERLAY_P = 0.25, OVERLAY_SNR_DB = (5.0, 25.0),
    GAP_P = 0.35, GAP_SEC = (0.1, 2.5), GAP_DIGITAL_FRAC = 0.5,
    NOISE_FLOOR_P = 0.35, NOISE_FLOOR_DBFS = (-75.0, -50.0),
    NOISE_GATE_P = 0.15,
    CLIP_P = 0.08,
    NONSPEECH_AS_REAL_P = 0.03,     # set 0 to disable
    # ---- splits for the new real speakers ----
    LS_VAL_FRAC = 0.15, LS_TEST_FRAC = 0.15,
    SEED = 1022,
    # ---- storage ----
    DRIVE_DIR  = "/content/drive/MyDrive/CAC_deepfake_detector",
    LOCAL_DIR  = "/content/work",
    ORIG_CACHE = "cache_en",        # the v2 notebook's cache folder (reused, not rebuilt)
    RUN_TAG    = "posttrain_v3",    # new folders: cache_extra_real_v3, ckpt_posttrain_v3, export_posttrain_v3
    OWN_DATA_SUBDIR = "own_data",
    ARCH_CONFIG_JSON = None,        # None = Wav2Vec2Config.from_pretrained(BASE_MODEL)
    # ---- quick end-to-end check (tiny data, a few steps). Leave False for real training. ----
    SMOKE_TEST = False,
)
import os, json
if os.environ.get("CAC_NB_OVERRIDES"):              # used only by the offline self-test of this notebook
    CFG.update(json.load(open(os.environ["CAC_NB_OVERRIDES"])))
assert "test-clean" not in CFG["LIBRISPEECH_SUBSETS"], "test-clean holds the benchmark's human controls"
print(json.dumps(CFG, indent=1))

## 2. GPU, Drive, and automatic batch/precision settings

In [ ]:
import sys, subprocess, torch

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
elif not os.environ.get("CAC_NB_OVERRIDES"):
    CFG["DRIVE_DIR"] = os.path.abspath("drive_out"); CFG["LOCAL_DIR"] = os.path.abspath("work")
os.makedirs(CFG["DRIVE_DIR"], exist_ok=True); os.makedirs(CFG["LOCAL_DIR"], exist_ok=True)

if torch.cuda.is_available():
    DEVICE = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    try:    USE_BF16 = torch.cuda.is_bf16_supported(including_emulation=False)
    except TypeError: USE_BF16 = torch.cuda.get_device_capability()[0] >= 8
    AMP_DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
    if   gpu_mem >= 70: BATCH, ACCUM = 32, 1
    elif gpu_mem >= 35: BATCH, ACCUM = 24, 1
    elif gpu_mem >= 20: BATCH, ACCUM = 16, 2
    else:               BATCH, ACCUM = 8, 4
    print(f"GPU: {gpu_name} ({gpu_mem:.0f} GB) | precision: {'bf16' if USE_BF16 else 'fp16'} | batch {BATCH} x accum {ACCUM}")
else:
    DEVICE, AMP_DTYPE, BATCH, ACCUM, gpu_mem, gpu_name = "cpu", None, 2, 1, 0, "cpu"
    print("⚠️ No GPU. Data preparation (sections 1-7) is fine on CPU; training refuses to run on CPU unless SMOKE_TEST=True.")
NUM_WORKERS = min(4, os.cpu_count() or 2)
print("CPU workers:", NUM_WORKERS)

## 3. Packages

In [ ]:
if IN_COLAB:
    %pip install -q soundfile scikit-learn matplotlib tqdm
import transformers, numpy as np
print("torch", torch.__version__, "| transformers", transformers.__version__, "| numpy", np.__version__)
def _ffmpeg_has(encoder):
    return subprocess.run(f"ffmpeg -hide_banner -encoders 2>/dev/null | grep -q {encoder}", shell=True).returncode == 0
HAS_OPUS, HAS_MP3 = _ffmpeg_has("libopus"), _ffmpeg_has("libmp3lame")
print("ffmpeg libopus:", HAS_OPUS, "| libmp3lame:", HAS_MP3)

## 4. Imports, seeds, folders

In [ ]:
import re, io, math, time, glob, random, shutil, tempfile, hashlib, tarfile, zipfile, urllib.request
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import pandas as pd, soundfile as sf
from scipy.signal import resample_poly, butter, sosfilt, lfilter
from tqdm.auto import tqdm
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.metrics import roc_curve, roc_auc_score

def seed_all(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
seed_all(CFG["SEED"])

SR = CFG["SR"]; WIN = int(CFG["TRAIN_SEC"] * SR)
DRIVE = Path(CFG["DRIVE_DIR"]); LOCAL = Path(CFG["LOCAL_DIR"])
sfx = "_smoke" if CFG["SMOKE_TEST"] else ""
ORIG_CACHE = DRIVE / CFG["ORIG_CACHE"]
EXTRA_CACHE = DRIVE / f"cache_extra_real_v3{sfx}"
CKPT_DIR = DRIVE / f"ckpt_{CFG['RUN_TAG']}{sfx}"
EXPORT = DRIVE / f"export_{CFG['RUN_TAG']}{sfx}"
for d in (EXTRA_CACHE, CKPT_DIR, EXPORT, LOCAL): d.mkdir(parents=True, exist_ok=True)

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""): h.update(b)
    return h.hexdigest()

def download(urls, dest, desc):
    # Resumable-enough single-file download with mirror fallback (re-run the cell after a failure).
    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0: return dest
    tmp = dest.with_suffix(dest.suffix + ".part"); last = None
    for url in ([urls] if isinstance(urls, str) else urls):
        for attempt in range(3):
            try:
                with urllib.request.urlopen(url, timeout=60) as r, open(tmp, "wb") as f, \
                     tqdm(total=int(r.headers.get("Content-Length") or 0), unit="B", unit_scale=True, desc=desc, mininterval=2) as bar:
                    for chunk in iter(lambda: r.read(1 << 20), b""):
                        f.write(chunk); bar.update(len(chunk))
                tmp.rename(dest); return dest
            except Exception as e:
                last = e; print(f"  {url}: {type(e).__name__}: {str(e)[:120]} (attempt {attempt+1})"); time.sleep(3 * (attempt + 1))
    raise RuntimeError(f"Could not download {desc}: {last}")

## 5. Load the v2 cache (MLAAD fakes + M-AILABS real) — reused, not rebuilt
If this fails, run cells 1–6 of the v2 notebook once; they build `cache_en` on Drive.

In [ ]:
if not ((ORIG_CACHE / "manifest.csv").exists() and (ORIG_CACHE / "audio_int16.npy").exists()):
    raise SystemExit(f"v2 cache not found in {ORIG_CACHE}. Run the v2 notebook's sections 1-6 first (or set ORIG_CACHE).")
orig_blob_local = LOCAL / "orig_audio_int16.npy"
if not orig_blob_local.exists():
    print("copying v2 cache to local disk (faster reads during training)...")
    shutil.copy(ORIG_CACHE / "audio_int16.npy", orig_blob_local)
BLOBS = [np.load(orig_blob_local, mmap_mode="r")]
orig = pd.read_csv(ORIG_CACHE / "manifest.csv")
orig["blob"] = 0
orig["source"] = np.where(orig.label == 1, "tts:" + orig.system.astype(str), "mailabs:" + orig.group.astype(str))
orig["corpus"] = np.where(orig.label == 1, "MLAAD", "M-AILABS")
print(orig.groupby(["split", "label"]).size().unstack().rename(columns={0: "real", 1: "fake"}))
print("Note: v2 split M-AILABS real clips by book, not by speaker, so its real val/test clips share speakers with training.\n"
      "The new LibriSpeech speakers below are split by speaker and give the cleaner real-speech numbers.")

## 6. Extra real speech: LibriSpeech (split by speaker), cached to Drive
About 1 GB of downloads. Same preprocessing as v2 (16 kHz, edge-silence trim, peak-normalize).

In [ ]:
def load_audio(path, sr=SR):
    try:
        x, s = sf.read(path, dtype="float32", always_2d=True)
    except Exception:
        # mp3/m4a/other: decode with ffmpeg to 16 kHz mono float32
        r = subprocess.run(["ffmpeg", "-loglevel", "quiet", "-i", str(path), "-f", "f32le", "-ac", "1", "-ar", str(sr), "-"],
                           capture_output=True, check=True)
        return np.frombuffer(r.stdout, dtype=np.float32).copy()
    x = x.mean(axis=1)
    if s != sr:
        g = math.gcd(int(s), sr); x = resample_poly(x, sr // g, int(s) // g).astype(np.float32)
    return x

def trim_silence(x, sr=SR, top_db=40.0, frame=0.02):   # identical to v2
    n = int(sr * frame)
    if len(x) < n * 2: return x
    k = len(x) // n
    e = 10 * np.log10(np.mean(x[: k * n].reshape(k, n) ** 2, axis=1) + 1e-12)
    keep = np.where(e > e.max() - top_db)[0]
    if len(keep) == 0: return x
    return x[keep[0] * n : (keep[-1] + 1) * n]

def preprocess_clip(path):                            # identical to v2's proc()
    try:
        x = trim_silence(load_audio(path))
        if len(x) < CFG["MIN_SEC"] * SR: return None
        x = x[: int(CFG["MAX_STORE_SEC"] * SR)]
        peak = np.abs(x).max()
        if peak < 1e-4: return None
        return (x / peak * 0.9 * 32767).astype(np.int16)
    except Exception:
        return None

def build_cache(items, cache_dir, desc):
    raw = LOCAL / f"{desc}.raw"; kept, lengths = [], []
    with open(raw, "wb") as fh, ThreadPoolExecutor(NUM_WORKERS * 2) as ex, tqdm(total=len(items), desc=desc) as bar:
        for s in range(0, len(items), 256):
            chunk = items[s:s + 256]
            for it, a in zip(chunk, ex.map(lambda it: preprocess_clip(it["path"]), chunk)):
                if a is not None:
                    fh.write(a.tobytes()); kept.append(it); lengths.append(len(a))
            bar.update(len(chunk))
    if not kept: raise RuntimeError(f"No usable audio for {desc}")
    lengths = np.array(lengths, np.int64)
    out = np.lib.format.open_memmap(LOCAL / f"{desc}.npy", mode="w+", dtype=np.int16, shape=(int(lengths.sum()),))
    src = np.memmap(raw, dtype=np.int16, mode="r")
    for s in range(0, len(out), 1 << 26): out[s:s + (1 << 26)] = src[s:s + (1 << 26)]
    out.flush(); del out, src; raw.unlink()
    m = pd.DataFrame(kept); m["offset"] = np.concatenate([[0], np.cumsum(lengths)[:-1]]); m["length"] = lengths
    shutil.copy(LOCAL / f"{desc}.npy", cache_dir / "audio_int16.npy"); m.to_csv(cache_dir / "manifest.csv", index=False)
    return m

BENCHMARK_SPEAKERS = {"1320", "6930"}                # LibriSpeech test-clean controls used by the repo benchmark
if (EXTRA_CACHE / "manifest.csv").exists() and (EXTRA_CACHE / "audio_int16.npy").exists():
    extra = pd.read_csv(EXTRA_CACHE / "manifest.csv", dtype={"speaker": str})
    print("LibriSpeech cache found on Drive:", len(extra), "clips")
else:
    root = Path(CFG["LIBRISPEECH_LOCAL_DIR"]) if CFG["LIBRISPEECH_LOCAL_DIR"] else LOCAL / "librispeech"
    root.mkdir(parents=True, exist_ok=True)
    for subset in CFG["LIBRISPEECH_SUBSETS"]:
        if (root / "LibriSpeech" / subset).exists(): continue
        tgz = download([f"{m}/{subset}.tar.gz" for m in CFG["LIBRISPEECH_MIRRORS"]], LOCAL / f"{subset}.tar.gz", subset)
        with tarfile.open(tgz) as t:
            t.extractall(root, filter="data") if hasattr(tarfile, "data_filter") else t.extractall(root)
        tgz.unlink()
    rng = random.Random(CFG["SEED"]); items = []
    for subset in CFG["LIBRISPEECH_SUBSETS"]:
        for spk_dir in sorted((root / "LibriSpeech" / subset).iterdir()):
            if not spk_dir.is_dir(): continue
            assert spk_dir.name not in BENCHMARK_SPEAKERS, "benchmark speaker found in training data"
            files = sorted(spk_dir.rglob("*.flac"))
            for p in rng.sample(files, min(len(files), CFG["MAX_UTTS_PER_SPEAKER"] if not CFG["SMOKE_TEST"] else 5)):
                items.append(dict(path=str(p), subset=subset, speaker=spk_dir.name))
    print(f"{len(items)} LibriSpeech utterances from {len({i['speaker'] for i in items})} speakers")
    extra = build_cache(items, EXTRA_CACHE, "librispeech")
    extra["speaker"] = extra["speaker"].astype(str)
    speakers = sorted(extra.speaker.unique()); random.Random(CFG["SEED"]).shuffle(speakers)
    n_te, n_va = round(len(speakers) * CFG["LS_TEST_FRAC"]), round(len(speakers) * CFG["LS_VAL_FRAC"])
    te, va = set(speakers[:n_te]), set(speakers[n_te:n_te + n_va])
    extra["split"] = np.where(extra.speaker.isin(te), "test", np.where(extra.speaker.isin(va), "val", "train"))
    extra.to_csv(EXTRA_CACHE / "manifest.csv", index=False)
    print("Cached to", EXTRA_CACHE)

extra_blob_local = LOCAL / "librispeech.npy"
if not extra_blob_local.exists(): shutil.copy(EXTRA_CACHE / "audio_int16.npy", extra_blob_local)
BLOBS.append(np.load(extra_blob_local, mmap_mode="r"))
extra = extra.assign(label=0, system="bonafide_librispeech", group="ls:" + extra.speaker.astype(str), blob=1,
                     source="librispeech:" + extra.speaker.astype(str), corpus="LibriSpeech")
cols = ["path", "label", "system", "group", "offset", "length", "split", "blob", "source", "corpus"]
manifest = pd.concat([orig[cols], extra[cols]], ignore_index=True)
assert not set(extra.speaker.astype(str)) & BENCHMARK_SPEAKERS
for s in ("train", "val", "test"):          # speakers never cross splits
    others = set(extra.loc[extra.split != s, "speaker"]); assert not (set(extra.loc[extra.split == s, "speaker"]) & others)
print(manifest.groupby(["split", "corpus"]).size().unstack(fill_value=0))

## 7. Background-sound banks: recorded (ESC-50), synthetic music, DTMF, babble

In [ ]:
def esc50_bank():
    if not CFG["USE_ESC50"]: return []
    base = Path(CFG["ESC50_LOCAL_DIR"]) if CFG["ESC50_LOCAL_DIR"] else LOCAL / "esc50"
    audio_dir = next(iter(base.glob("*/audio")), None) if base.exists() else None
    if audio_dir is None:
        z = download(CFG["ESC50_URL"], LOCAL / "esc50.zip", "ESC-50")
        with zipfile.ZipFile(z) as f: f.extractall(base)
        z.unlink(); audio_dir = next(iter(base.glob("*/audio")))
    files = sorted(audio_dir.glob("*.wav")); random.Random(CFG["SEED"]).shuffle(files)
    out = []
    for p in tqdm(files[: CFG["ESC50_MAX_CLIPS"] if not CFG["SMOKE_TEST"] else 20], desc="ESC-50"):
        x = load_audio(str(p))
        if np.abs(x).max() > 1e-3: out.append((x / (np.abs(x).max() + 1e-9) * 0.9 * 32767).astype(np.int16))
    return out
try:
    NOISE_BANK = esc50_bank()
except Exception as e:
    NOISE_BANK = []; print("⚠️ ESC-50 unavailable, using synthetic noise only:", e)
print("recorded noise clips:", len(NOISE_BANK))

def synth_music(n, rng):
    t = np.arange(n) / SR
    if rng.random() < 0.5:                                    # sustained chord with tremolo
        root = rng.uniform(110, 330); ratios = [1, 1.26, 1.5, 2][: rng.integers(2, 5)]
        x = sum(np.sin(2 * np.pi * root * r * t + rng.uniform(0, 6.3)) for r in ratios)
        x = x * (0.6 + 0.4 * np.sin(2 * np.pi * rng.uniform(0.5, 4) * t))
    else:                                                     # melody of harmonic notes
        note = int(SR * rng.uniform(0.15, 0.5)); x = np.zeros(n)
        for s in range(0, n, note):
            f = 220 * 2 ** (rng.integers(0, 15) / 12); m = min(note, n - s); tt = np.arange(m) / SR
            x[s:s + m] = sum(np.sin(2 * np.pi * k * f * tt) / k for k in range(1, 6)) * np.exp(-tt * rng.uniform(1, 6))
    return x.astype(np.float32)

DTMF = [(697, 1209), (697, 1336), (697, 1477), (770, 1209), (770, 1336), (770, 1477), (852, 1209), (852, 1336), (852, 1477), (941, 1336)]
def synth_dtmf(n, rng):
    t = np.arange(n) / SR; x = np.zeros(n); s = 0
    while s < n:
        on, off = int(SR * rng.uniform(.08, .5)), int(SR * rng.uniform(.05, .4)); a, b = DTMF[rng.integers(len(DTMF))]
        e = min(n, s + on); x[s:e] = np.sin(2 * np.pi * a * t[s:e]) + np.sin(2 * np.pi * b * t[s:e]); s = e + off
    return x.astype(np.float32)

def colored_noise(n, rng, kind=None):
    w = rng.standard_normal(n).astype(np.float32); kind = kind or rng.choice(["white", "pink", "brown"])
    if kind == "brown": w = np.cumsum(w); w -= np.linspace(w[0], w[-1], n)
    elif kind == "pink": w = lfilter([0.049922035, -0.095993537, 0.050612699, -0.004408786], [1, -2.494956002, 2.017265875, -0.522189400], w)
    return (w / (w.std() + 1e-9)).astype(np.float32)

def recorded_noise(n, rng):
    if not NOISE_BANK: return colored_noise(n, rng)
    x = NOISE_BANK[rng.integers(len(NOISE_BANK))].astype(np.float32) / 32768
    if len(x) < n: x = np.tile(x, int(np.ceil(n / len(x))))
    s = rng.integers(0, len(x) - n + 1); return x[s:s + n]

TRAIN_REAL_IDX = None   # filled after the splits exist (babble uses only training real speech)
def babble(n, rng):
    x = np.zeros(n, np.float32)
    for _ in range(rng.integers(2, 5)):
        r = manifest.iloc[TRAIN_REAL_IDX[rng.integers(len(TRAIN_REAL_IDX))]]
        a = np.asarray(BLOBS[int(r.blob)][int(r.offset):int(r.offset) + int(r.length)], np.float32) / 32768
        x += fit_window_n(a, n, rng)
    return x

def fit_window_n(x, n, rng=None):
    if len(x) < n: x = np.tile(x, int(np.ceil(n / max(len(x), 1))))[:n] if len(x) else np.zeros(n, np.float32)
    elif len(x) > n:
        s = rng.integers(0, len(x) - n + 1) if rng is not None else (len(x) - n) // 2; x = x[s:s + n]
    return x.astype(np.float32)
print("banks ready")

## 8. Augmentation (the training function never receives the label)
`phone_channel` is v2's telephone simulation: band-limit, μ-law, packet loss, plus optional Opus and MP3. Noise is now applied separately, over a wider range.

In [ ]:
_SOS_PHONE = butter(4, [300, 3400], btype="bandpass", fs=8000, output="sos")

def mulaw_roundtrip(x, mu=255):
    x = np.clip(x, -1, 1); y = np.sign(x) * np.log1p(mu * np.abs(x)) / np.log1p(mu)
    q = np.round((y + 1) / 2 * mu) / mu * 2 - 1
    return (np.sign(q) * ((1 + mu) ** np.abs(q) - 1) / mu).astype(np.float32)

def codec_roundtrip(x8k, codec, bitrate):
    with tempfile.TemporaryDirectory() as d:
        a, b, c = f"{d}/a.wav", f"{d}/b.{'ogg' if codec == 'opus' else 'mp3'}", f"{d}/c.wav"
        sf.write(a, x8k, 8000)
        enc = ["-c:a", "libopus"] if codec == "opus" else ["-c:a", "libmp3lame"]
        r1 = subprocess.run(["ffmpeg", "-loglevel", "quiet", "-y", "-i", a, *enc, "-b:a", bitrate, b])
        r2 = subprocess.run(["ffmpeg", "-loglevel", "quiet", "-y", "-i", b, "-ar", "8000", "-ac", "1", c]) if r1.returncode == 0 else r1
        if r2.returncode != 0: return x8k
        y, _ = sf.read(c, dtype="float32"); y = y[: len(x8k)]
        return np.pad(y, (0, len(x8k) - len(y))) if len(y) < len(x8k) else y

def phone_channel(x, rng):
    n = len(x); x = x * 10 ** (rng.uniform(-12, 3) / 20)
    y = resample_poly(x, 1, 2).astype(np.float32)
    if rng.random() < 0.8: y = sosfilt(_SOS_PHONE, y).astype(np.float32)
    u = rng.random()
    if HAS_OPUS and u < CFG["OPUS_P"]: y = codec_roundtrip(y, "opus", rng.choice(["8k", "12k", "16k"]))
    elif HAS_MP3 and u < CFG["OPUS_P"] + CFG["MP3_P"]: y = codec_roundtrip(y, "mp3", rng.choice(["16k", "24k", "32k"]))
    if rng.random() < 0.85: y = mulaw_roundtrip(y)
    if rng.random() < 0.3:
        fl = 160; k = len(y) // fl; y[: k * fl][np.repeat(rng.random(k) < 0.02, fl)] = 0
    y = resample_poly(y, 2, 1).astype(np.float32)[:n]
    return np.pad(y, (0, n - len(y))) if len(y) < n else y

def mix_at_snr(x, noise, snr_db):
    p = np.mean(x ** 2) + 1e-10; q = np.mean(noise ** 2) + 1e-10
    return (x + noise * np.sqrt(p / q / 10 ** (snr_db / 10))).astype(np.float32)

def background(n, rng):
    u = rng.random()
    return synth_music(n, rng) if u < .35 else synth_dtmf(n, rng) if u < .5 else babble(n, rng) if u < .75 else recorded_noise(n, rng)

def insert_gaps(x, rng, digital=None):
    x = x.copy(); n = len(x)
    for _ in range(rng.integers(1, 3)):
        L = min(n - 1, int(SR * rng.uniform(*CFG["GAP_SEC"])))
        s = rng.choice([0, n - L, rng.integers(0, n - L + 1)])          # edges included: the live failure was a leading gap
        dig = (rng.random() < CFG["GAP_DIGITAL_FRAC"]) if digital is None else digital
        x[s:s + L] = 0 if dig else colored_noise(L, rng) * 10 ** (rng.uniform(*CFG["NOISE_FLOOR_DBFS"]) / 20)
    return x

def noise_gate(x, rng):
    fl = int(SR * .02); k = len(x) // fl
    if k == 0: return x
    e = np.sqrt(np.mean(x[: k * fl].reshape(k, fl) ** 2, 1) + 1e-12)
    thr = np.percentile(e, rng.uniform(20, 50))
    x = x.copy(); x[: k * fl][np.repeat(e <= thr, fl)] = 0; return x

def train_augment(x, rng):
    # Called identically for real and fake clips; it has no access to the label.
    n = len(x)
    if rng.random() < CFG["OVERLAY_P"]: x = mix_at_snr(x, background(n, rng), rng.uniform(*CFG["OVERLAY_SNR_DB"]))
    if rng.random() < CFG["PHONE_AUG_P"]: x = phone_channel(x, rng)
    if rng.random() < CFG["NOISE_P"]:
        noise = recorded_noise(n, rng) if rng.random() < CFG["RECORDED_NOISE_FRAC"] else colored_noise(n, rng)
        x = mix_at_snr(x, noise, rng.uniform(*CFG["NOISE_SNR_DB"]))
    if rng.random() < CFG["NOISE_FLOOR_P"]:
        x = x + colored_noise(n, rng) * 10 ** (rng.uniform(*CFG["NOISE_FLOOR_DBFS"]) / 20)
    if rng.random() < CFG["NOISE_GATE_P"]: x = noise_gate(x, rng)
    if rng.random() < CFG["GAP_P"]: x = insert_gaps(x, rng)
    if rng.random() < CFG["CLIP_P"]:
        c = np.abs(x).max() * rng.uniform(.3, .7); x = np.clip(x, -c, c)
    return np.clip(x, -1, 1).astype(np.float32)

def nonspeech_window(n, rng):
    u = rng.random()
    x = synth_music(n, rng) if u < .25 else synth_dtmf(n, rng) if u < .45 else recorded_noise(n, rng) if u < .75 else colored_noise(n, rng)
    x = x / (np.abs(x).max() + 1e-9) * rng.uniform(.01, .9)
    return insert_gaps(x, rng) if rng.random() < .3 else x

# Deterministic evaluation conditions. "gap2s" reproduces the benchmark failure (2 s exact zeros + 2 s speech).
EVAL_CONDITIONS = ["clean", "phone", "noise20", "noise10", "phone_noise10", "gap2s", "music10"]
def eval_condition(x, cond, rng):
    n = len(x)
    if cond == "clean": return x
    if cond == "phone": return np.clip(mix_at_snr(phone_channel(x, rng), colored_noise(n, rng, "white"), 25), -1, 1)
    if cond == "noise20": return mix_at_snr(x, colored_noise(n, rng, "white"), 20)
    if cond == "noise10": return mix_at_snr(x, colored_noise(n, rng, "white"), 10)
    if cond == "phone_noise10": return mix_at_snr(phone_channel(x, rng), colored_noise(n, rng, "white"), 10)
    if cond == "gap2s": y = x.copy(); y[: SR * 2] = 0; return y
    if cond == "music10":
        m = synth_music(n, np.random.default_rng(7)); return mix_at_snr(x, m, 10)
    raise ValueError(cond)

def normalize(x):                                         # same per-window z-normalization as inference
    return ((x - x.mean()) / (x.std() + 1e-7)).astype(np.float32)

_t = np.random.default_rng(0).standard_normal(WIN).astype(np.float32) * .1
for c in EVAL_CONDITIONS: assert eval_condition(_t, c, np.random.default_rng(1)).shape == (WIN,)
print("augmentation OK")

## 9. Datasets and loaders

In [ ]:
class AudioDS(Dataset):
    def __init__(self, df, train=False, condition="clean", seed=0):
        self.df = df.reset_index(drop=True); self.train, self.cond, self.seed = train, condition, seed
        self.blob, self.off, self.len, self.lab = (self.df[c].to_numpy() for c in ["blob", "offset", "length", "label"])
        self.rng = np.random.default_rng(seed)
    def __len__(self): return len(self.df)
    def clip(self, i):
        o, n = int(self.off[i]), int(self.len[i])
        return np.asarray(BLOBS[int(self.blob[i])][o:o + n], dtype=np.float32) / 32768.0
    def __getitem__(self, i):
        if self.train:
            rng = self.rng
            if rng.random() < CFG["NONSPEECH_AS_REAL_P"]:
                return torch.from_numpy(normalize(nonspeech_window(WIN, rng))), torch.tensor(0.0)
            x = train_augment(fit_window_n(self.clip(i), WIN, rng), rng)
        else:
            rng = np.random.default_rng(self.seed * 100003 + i)
            x = eval_condition(fit_window_n(self.clip(i), WIN), self.cond, rng)
        return torch.from_numpy(normalize(x)), torch.tensor(float(self.lab[i]))

def worker_init(_):
    info = torch.utils.data.get_worker_info()
    info.dataset.rng = np.random.default_rng(torch.initial_seed() % 2**32)

df = manifest.copy()
if CFG["SMOKE_TEST"]:
    df = pd.concat([d.head(24) for _, d in df.groupby(["split", "label", "corpus"])])
train_df, val_df, test_df = (df[df.split == s] for s in ["train", "val", "test"])
TRAIN_REAL_IDX = np.flatnonzero(((manifest.split == "train") & (manifest.label == 0)).to_numpy())

# Balance classes (as v2), and within real speech give M-AILABS and LibriSpeech equal total weight,
# so the new speakers aren't drowned out and the model can't learn "M-AILABS voice = real".
lab, corp = train_df.label.to_numpy(), train_df.corpus.to_numpy()
w = np.where(lab == 1, 1.0 / max(1, int((lab == 1).sum())), 0.0)
real_corpora = np.unique(corp[lab == 0])
for c in real_corpora:
    m = (lab == 0) & (corp == c); w[m] = 1.0 / (len(real_corpora) * m.sum())
print("sampling weight per class:", {int(k): float(w[lab == k].sum()) for k in (0, 1)},
      "| real corpora:", {c: float(w[(lab == 0) & (corp == c)].sum()) for c in real_corpora})
sampler = WeightedRandomSampler(torch.tensor(w, dtype=torch.double), num_samples=len(train_df), replacement=True)
dl_kw = dict(num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"), worker_init_fn=worker_init, persistent_workers=NUM_WORKERS > 0)
train_dl = DataLoader(AudioDS(train_df, train=True, seed=CFG["SEED"]), batch_size=BATCH, sampler=sampler, drop_last=True, **dl_kw)
def eval_loader(frame, cond, seed=1):
    return DataLoader(AudioDS(frame, condition=cond, seed=seed), batch_size=BATCH * 2,
                      num_workers=NUM_WORKERS, pin_memory=(DEVICE == "cuda"))
print(f"train {len(train_df)} | val {len(val_df)} | test {len(test_df)} | steps/epoch {len(train_dl)//ACCUM}")
xb, yb = next(iter(train_dl)); print("batch:", tuple(xb.shape), yb.tolist()[:8])

## 10. Model, loaded from the existing checkpoint

In [ ]:
from transformers import Wav2Vec2Config, Wav2Vec2Model

class AttnStatsPool(nn.Module):
    def __init__(self, d):
        super().__init__(); self.att = nn.Sequential(nn.Linear(d, 128), nn.Tanh(), nn.Linear(128, 1))
    def forward(self, h):
        a = torch.softmax(self.att(h).float(), dim=1).to(h.dtype); mu = (a * h).sum(1)
        sd = torch.sqrt(((a * (h - mu.unsqueeze(1)) ** 2).sum(1)).clamp(min=1e-6)); return torch.cat([mu, sd], dim=-1)

class DeepfakeDetector(nn.Module):                      # identical to v2 (state dicts are interchangeable)
    def __init__(self, keep_layers):
        super().__init__()
        cfg = (Wav2Vec2Config.from_json_file(CFG["ARCH_CONFIG_JSON"]) if CFG["ARCH_CONFIG_JSON"]
               else Wav2Vec2Config.from_pretrained(CFG["BASE_MODEL"]))
        cfg.num_hidden_layers = keep_layers; cfg.layerdrop = 0.0; cfg.mask_time_prob = 0.05; cfg.mask_feature_prob = 0.0
        self.backbone = Wav2Vec2Model(cfg)
        for p in self.backbone.feature_extractor.parameters(): p.requires_grad = False
        d = cfg.hidden_size
        self.layer_w = nn.Parameter(torch.zeros(keep_layers + 1))
        self.proj = nn.Sequential(nn.LayerNorm(d), nn.Linear(d, 256), nn.GELU()); self.pool = AttnStatsPool(256)
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(512, 128), nn.GELU(), nn.Dropout(0.2), nn.Linear(128, 1))
    def forward(self, wav):
        hs = self.backbone(wav, output_hidden_states=True).hidden_states; w = torch.softmax(self.layer_w, 0)
        return self.head(self.pool(self.proj(sum(w[i] * hs[i] for i in range(len(hs)))))).squeeze(-1)

if DEVICE == "cpu" and not CFG["SMOKE_TEST"]:
    raise SystemExit("Stop: switch to a GPU runtime before training (or set SMOKE_TEST=True for a CPU check).")

init_path = Path(CFG["INIT_CHECKPOINT"])
if not init_path.exists(): raise SystemExit(f"INIT_CHECKPOINT not found: {init_path}")
INIT_SHA = sha256(init_path)
print("init checkpoint", init_path, "sha256", INIT_SHA)
if CFG["EXPECTED_INIT_SHA256"] and INIT_SHA != CFG["EXPECTED_INIT_SHA256"]:
    print("⚠️ This is not byte-identical to the deployed best.pt. Fine if it holds the same weights in another wrapper "
          "(e.g. detector_state.pt); otherwise point INIT_CHECKPOINT at the deployed file.")
state = torch.load(init_path, map_location="cpu", weights_only=False)
state = state["model"] if isinstance(state, dict) and "model" in state else state
model = DeepfakeDetector(CFG["KEEP_LAYERS"])
model.load_state_dict(state)
model.to(DEVICE)
INIT_STATE = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}   # kept for the before/after report
if DEVICE == "cuda" and gpu_mem < 20:
    model.backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
print(f"params {sum(p.numel() for p in model.parameters())/1e6:.1f}M, trainable {sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6:.1f}M")

## 11. Robustness evaluation helpers, and the "before" report
All scores are **raw logits in float64**. The v2 thresholds came from float32 sigmoids, which saturate.

In [ ]:
@torch.no_grad()
def predict_logits(dl):
    model.eval(); Z, Y = [], []
    for x, y in dl:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", dtype=AMP_DTYPE or torch.float16, enabled=AMP_DTYPE is not None):
            Z.append(model(x).float().cpu().double()); Y.append(y)
    return torch.cat(Y).numpy(), torch.cat(Z).numpy()

def eer_and_threshold(y, z):
    fpr, tpr, thr = roc_curve(y, z); fnr = 1 - tpr; i = np.nanargmin(np.abs(fnr - fpr))
    return float((fpr[i] + fnr[i]) / 2), float(thr[i])

def threshold_at_fpr(y, z, target):
    real = np.sort(z[y == 0])[::-1]; k = int(np.floor(target * len(real)))   # at most `target` of real windows at or above
    return float(np.nextafter(real[k], np.inf)) if k < len(real) else float(real.min())

def robustness(frame, tag, conditions=EVAL_CONDITIONS, seed=1):
    rows, pooled_y, pooled_z = [], [], []
    for c in conditions:
        y, z = predict_logits(eval_loader(frame, c, seed)); pooled_y.append(y); pooled_z.append(z)
        eer, _ = eer_and_threshold(y, z)
        rows.append(dict(set=tag, condition=c, eer=eer, auc=roc_auc_score(y, z),
                         real_median=float(np.median(z[y == 0])), fake_median=float(np.median(z[y == 1])),
                         real_p95=float(np.percentile(z[y == 0], 95)), fake_p05=float(np.percentile(z[y == 1], 5))))
    y, z = np.concatenate(pooled_y), np.concatenate(pooled_z)
    eer, thr = eer_and_threshold(y, z)
    rows.append(dict(set=tag, condition="POOLED (one threshold)", eer=eer, auc=roc_auc_score(y, z),
                     real_median=float(np.median(z[y == 0])), fake_median=float(np.median(z[y == 1])),
                     real_p95=float(np.percentile(z[y == 0], 95)), fake_p05=float(np.percentile(z[y == 1], 5))))
    return pd.DataFrame(rows), (y, z)

def nonspeech_probe(n_each=40, seed=5):
    rng = np.random.default_rng(seed); out = {}
    kinds = {"dtmf": synth_dtmf, "music": synth_music, "white_noise": lambda n, r: colored_noise(n, r, "white"),
             "recorded_noise": recorded_noise}
    for name, fn in kinds.items():
        x = np.stack([normalize(fn(WIN, rng) * .3) for _ in range(n_each)])
        with torch.no_grad(), torch.autocast("cuda", dtype=AMP_DTYPE or torch.float16, enabled=AMP_DTYPE is not None):
            z = np.concatenate([model(torch.from_numpy(x[i:i + 16]).to(DEVICE)).float().cpu().double().numpy() for i in range(0, len(x), 16)])
        out[name] = dict(median=float(np.median(z)), max=float(z.max()))
    return out

before_path = CKPT_DIR / "before_report.json"
if before_path.exists():
    BEFORE = json.loads(before_path.read_text()); print("loaded saved 'before' report")
else:
    print("Scoring the starting checkpoint on the robustness conditions (one-off, cached to Drive)...")
    bdf, _ = robustness(val_df, "val (before)")
    BEFORE = dict(val=bdf.to_dict("records"), nonspeech=nonspeech_probe())
    before_path.write_text(json.dumps(BEFORE, indent=1))
print(pd.DataFrame(BEFORE["val"]).round(3).to_string(index=False))
print("non-speech logits (before):", json.dumps(BEFORE["nonspeech"], indent=0))

## 12. Fine-tune (auto-resumes from Drive). Best epoch = lowest pooled validation EER

In [ ]:
head_params = [p for n, p in model.named_parameters() if not n.startswith("backbone.") and p.requires_grad]
bb_params   = [p for n, p in model.named_parameters() if n.startswith("backbone.") and p.requires_grad]
opt = torch.optim.AdamW([{"params": bb_params, "lr": CFG["LR_BACKBONE"]}, {"params": head_params, "lr": CFG["LR_HEAD"]}],
                        weight_decay=CFG["WEIGHT_DECAY"])
steps_per_epoch = max(1, len(train_dl) // ACCUM)
epochs = 1 if CFG["SMOKE_TEST"] else CFG["EPOCHS"]
total_steps = steps_per_epoch * epochs; warm = max(1, int(total_steps * CFG["WARMUP_FRAC"]))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total_steps))))
scaler = torch.amp.GradScaler("cuda", enabled=(AMP_DTYPE == torch.float16))
loss_fn = nn.BCEWithLogitsLoss()
# Validation for model selection uses a subset of conditions to keep each epoch's check short.
SELECT_CONDITIONS = ["clean", "phone", "noise10", "gap2s", "music10"]

start_epoch, best_score, bad_epochs, history = 0, float("inf"), 0, []
last_ckpt, best_ckpt = CKPT_DIR / "last.pt", CKPT_DIR / "best.pt"
if last_ckpt.exists():
    ck = torch.load(last_ckpt, map_location=DEVICE, weights_only=False)
    model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); sched.load_state_dict(ck["sched"])
    if ck.get("scaler"): scaler.load_state_dict(ck["scaler"])
    start_epoch, best_score, bad_epochs, history = ck["epoch"] + 1, ck["best_score"], ck["bad_epochs"], ck["history"]
    print(f"Resumed after epoch {ck['epoch']+1} (best pooled val EER {best_score:.4f})")
elif not best_ckpt.exists():
    # Epoch 0 = the starting checkpoint, so fine-tuning is only kept if it actually improves.
    before_pooled = next(r for r in BEFORE["val"] if r["condition"].startswith("POOLED"))
    best_score = before_pooled["eer"]
    torch.save({"model": INIT_STATE, "pooled_val_eer": best_score, "epoch": 0, "cfg": CFG, "init_sha256": INIT_SHA}, best_ckpt)
    print(f"starting point pooled val EER (all conditions): {best_score:.4f}")

for epoch in range(start_epoch, epochs):
    if bad_epochs >= CFG["PATIENCE"]: print("Early stopping."); break
    model.train(); t0 = time.time(); run = 0.0; opt.zero_grad(set_to_none=True)
    pbar = tqdm(enumerate(train_dl), total=len(train_dl), desc=f"epoch {epoch+1}/{epochs}")
    for i, (x, y) in pbar:
        x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE)
        with torch.autocast("cuda", dtype=AMP_DTYPE or torch.float16, enabled=AMP_DTYPE is not None):
            loss = loss_fn(model(x).float(), y) / ACCUM
        scaler.scale(loss).backward()
        if (i + 1) % ACCUM == 0:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), CFG["GRAD_CLIP"])
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
        run = 0.98 * run + 0.02 * loss.item() * ACCUM if i else loss.item() * ACCUM
        if i % 20 == 0: pbar.set_postfix(loss=f"{run:.4f}", lr=f"{sched.get_last_lr()[0]:.2e}")
        if CFG["SMOKE_TEST"] and i >= 6: break
    rdf, _ = robustness(val_df, f"val e{epoch+1}", SELECT_CONDITIONS)
    pooled = float(rdf.iloc[-1].eer); worst = float(rdf.iloc[:-1].eer.max())
    history.append(dict(epoch=epoch + 1, loss=run, pooled_val_eer=pooled, worst_condition_eer=worst,
                        **{f"eer_{r.condition}": r.eer for r in rdf.iloc[:-1].itertuples()}, minutes=(time.time() - t0) / 60))
    print(rdf.round(3).to_string(index=False))
    print(f"epoch {epoch+1}: loss {run:.4f} | pooled val EER {pooled*100:.2f}% | worst condition {worst*100:.2f}% | {(time.time()-t0)/60:.1f} min")
    if pooled < best_score:
        best_score, bad_epochs = pooled, 0
        torch.save({"model": model.state_dict(), "pooled_val_eer": pooled, "epoch": epoch + 1, "cfg": CFG, "init_sha256": INIT_SHA}, best_ckpt)
        print("  ✓ new best saved to Drive")
    else:
        bad_epochs += 1
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(), "scaler": scaler.state_dict(),
                "epoch": epoch, "best_score": best_score, "bad_epochs": bad_epochs, "history": history}, last_ckpt)
print(pd.DataFrame(history).round(4).to_string(index=False))

## 13. Final evaluation: thresholds (float64 logits), robustness before/after, unseen TTS systems, non-speech
Thresholds are chosen on **validation** windows pooled over every condition (one threshold for everything, as live), then reported on **test**.

In [ ]:
import matplotlib.pyplot as plt
best = torch.load(best_ckpt, map_location=DEVICE, weights_only=False)
model.load_state_dict(best["model"])
KEPT_EPOCH = best["epoch"]
print("using epoch", KEPT_EPOCH, "(0 = fine-tuning did not beat the starting checkpoint)")

val_rob, (yv, zv) = robustness(val_df, "val (after)")
LOGIT_EER_T = eer_and_threshold(yv, zv)[1]
LOGIT_1PCT_T = threshold_at_fpr(yv, zv, 0.01)
test_rob, (yt, zt) = robustness(test_df, "test (after)", seed=7)
def rates(y, z, t): return dict(real_flagged=float((z[y == 0] >= t).mean()), fakes_caught=float((z[y == 1] >= t).mean()))
print(f"logit thresholds (val, pooled): EER {LOGIT_EER_T:.3f} | 1% FPR {LOGIT_1PCT_T:.3f}")
print("test at 1%-FPR threshold:", rates(yt, zt, LOGIT_1PCT_T), "| at EER threshold:", rates(yt, zt, LOGIT_EER_T))
for t in (LOGIT_EER_T, LOGIT_1PCT_T):
    if t >= 27.6: print("⚠️ threshold above the live engine's logit clamp (27.6); the agent would clamp it. Report this before deploying.")

cmp = pd.concat([pd.DataFrame(BEFORE["val"]), val_rob]).pivot_table(index="condition", columns="set", values="eer", sort=False)
print("\nValidation EER by condition, before vs after:\n", (cmp * 100).round(2).to_string())
print("\nTest (after):\n", test_rob.round(3).to_string(index=False))

# By real-speech corpus (LibriSpeech is split by speaker; M-AILABS val/test share speakers with training)
for corpus in ["LibriSpeech", "M-AILABS"]:
    frame = pd.concat([test_df[test_df.label == 1], test_df[(test_df.label == 0) & (test_df.corpus == corpus)]])
    if (frame.label == 0).any():
        y, z = predict_logits(eval_loader(frame, "phone", 7)); print(f"test/phone with real={corpus}: EER {eer_and_threshold(y, z)[0]*100:.2f}%")

t = test_df.copy(); _, t["z"] = predict_logits(eval_loader(test_df, "phone", 7))
per_sys = t[t.label == 1].groupby("system").z.apply(lambda s: (s >= LOGIT_1PCT_T).mean()).sort_values()
print("\nDetection rate per unseen TTS system (phone, 1%-FPR threshold):\n", (per_sys * 100).round(1).to_string())

NONSPEECH_AFTER = nonspeech_probe()
print("\nNon-speech logits  before -> after:")
for k in NONSPEECH_AFTER: print(f"  {k:15s} median {BEFORE['nonspeech'][k]['median']:7.2f} -> {NONSPEECH_AFTER[k]['median']:7.2f}   max {BEFORE['nonspeech'][k]['max']:7.2f} -> {NONSPEECH_AFTER[k]['max']:7.2f}")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
(cmp * 100).plot.bar(ax=ax[0]); ax[0].set_ylabel("val EER (%)"); ax[0].set_title("Robustness before vs after")
ax[1].hist(zt[yt == 0], 60, alpha=.6, label="real"); ax[1].hist(zt[yt == 1], 60, alpha=.6, label="fake")
ax[1].axvline(LOGIT_1PCT_T, c="k", ls="--", label="1% FPR"); ax[1].axvline(LOGIT_EER_T, c="gray", ls=":", label="EER")
ax[1].set_xlabel("logit (all test conditions pooled)"); ax[1].legend()
plt.tight_layout(); plt.savefig(EXPORT / "robustness.png", dpi=120); plt.show()

## 14. Your own recordings (evaluation only)
Scores every file under `own_data/real/<source>/` and `own_data/fake/<source>/` in non-overlapping 4 s windows, with the same normalization as the live agent, using the **starting** checkpoint and the **post-trained** one. Skipped if the folder doesn't exist.

In [ ]:
OWN = DRIVE / CFG["OWN_DATA_SUBDIR"]; OWN_REPORT = None
def windows_of(x):
    k = len(x) // WIN
    return np.stack([normalize(x[i * WIN:(i + 1) * WIN]) for i in range(k)]) if k else np.zeros((0, WIN), np.float32)
@torch.no_grad()
def score_windows(state_dict, W):
    model.load_state_dict(state_dict); model.eval(); out = []
    for i in range(0, len(W), 16):
        with torch.autocast("cuda", dtype=AMP_DTYPE or torch.float16, enabled=AMP_DTYPE is not None):
            out.append(model(torch.from_numpy(W[i:i + 16]).to(DEVICE)).float().cpu().double().numpy())
    return np.concatenate(out) if out else np.zeros(0)
if OWN.exists():
    rows = []; after_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    for label in ("real", "fake"):
        for src in sorted(p for p in (OWN / label).glob("*") if p.is_dir()):
            W = [windows_of(load_audio(str(f))) for f in sorted(src.rglob("*")) if f.suffix.lower() in (".wav", ".flac", ".mp3", ".m4a", ".ogg")]
            W = np.concatenate(W) if W else np.zeros((0, WIN), np.float32)
            if not len(W): continue
            zb, za = score_windows(INIT_STATE, W), score_windows(after_state, W)
            rows.append(dict(label=label, source=src.name, windows=len(W), before_median=np.median(zb), after_median=np.median(za),
                             after_flagged_1pct=float((za >= LOGIT_1PCT_T).mean()), after_flagged_eer=float((za >= LOGIT_EER_T).mean())))
    model.load_state_dict(after_state)
    OWN_REPORT = pd.DataFrame(rows)
    print(OWN_REPORT.round(3).to_string(index=False) if len(OWN_REPORT) else "own_data has no audio")
else:
    print(f"No {OWN} folder; skipped. Add consented real/fake recordings there for an out-of-corpus check.")

## 15. Latency check

In [ ]:
model.eval(); x = torch.randn(1, WIN, device=DEVICE)
def bench(fn, n=30):
    for _ in range(3): fn()
    if DEVICE == "cuda": torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(n): fn()
    if DEVICE == "cuda": torch.cuda.synchronize()
    return (time.time() - t0) / n * 1000
with torch.no_grad():
    if DEVICE == "cuda":
        with torch.autocast("cuda", dtype=AMP_DTYPE): print(f"GPU ({gpu_name}): {bench(lambda: model(x)):.1f} ms per 4 s window")
    print("(Same architecture as v2, so CPU/MPS latency is unchanged from the deployed model.)")

## 16. Export a bundle the repo can load
Writes `detector_state.pt`, `detector_config.json` (with **logit thresholds**), `posttrain_report.json`, `robustness.png` and `SHA256SUMS.txt` to `MyDrive/CAC_deepfake_detector/export_posttrain_v3/`. Don't edit `detector_config.json` after download: the agent verifies its SHA-256.

In [ ]:
def sigmoid64(z): return float(1 / (1 + math.exp(-z))) if z >= 0 else float(math.exp(z) / (1 + math.exp(z)))
p_eer, p_1pct = sigmoid64(LOGIT_EER_T), sigmoid64(LOGIT_1PCT_T)
assert 0 < p_eer <= p_1pct < 1, "probability thresholds must be strictly below 1 (float64)"
test_by_cond = {r["condition"]: dict(eer=r["eer"], auc=r["auc"]) for r in test_rob.to_dict("records")}
meta = dict(
    base_model=CFG["BASE_MODEL"], keep_layers=CFG["KEEP_LAYERS"], sample_rate=SR, window_sec=CFG["TRAIN_SEC"],
    threshold_eer=p_eer, threshold_1pct_fpr=p_1pct,                       # float64 sigmoid of the logit thresholds
    logit_threshold_eer=LOGIT_EER_T, logit_threshold_1pct_fpr=LOGIT_1PCT_T,
    threshold_selection="validation windows pooled over all robustness conditions, raw float64 logits",
    val_eer=float(val_rob.iloc[-1].eer), val_eer_by_condition={r["condition"]: r["eer"] for r in val_rob.to_dict("records")},
    test=dict(clean=test_by_cond["clean"], phone=test_by_cond["phone"], **{k: v for k, v in test_by_cond.items() if k not in ("clean", "phone")}),
    unseen_test_systems=sorted(test_df.query("label==1").system.unique()),
    score_meaning="sigmoid(logit) = probability the audio is AI-generated (uncalibrated)",
    trained=time.strftime("%Y-%m-%d"),
    posttrain=dict(version="v3", init_checkpoint_sha256=INIT_SHA, kept_epoch=KEPT_EPOCH,
                   real_corpora=sorted(manifest.query("label==0").corpus.unique()), librispeech_subsets=CFG["LIBRISPEECH_SUBSETS"],
                   excluded_speakers=sorted(BENCHMARK_SPEAKERS), augmentation={k: CFG[k] for k in CFG if k.endswith(("_P", "_DB", "_SEC", "_DBFS", "_FRAC"))},
                   caveat="Benchmark LibriSpeech controls are now in-corpus (other speakers of the same corpus were trained on); use own recordings for an out-of-corpus check."),
)
torch.save({k: v.detach().cpu() for k, v in model.state_dict().items()}, EXPORT / "detector_state.pt")
(EXPORT / "detector_config.json").write_text(json.dumps(meta, indent=2))
report = dict(before=BEFORE, after=dict(val=val_rob.to_dict("records"), test=test_rob.to_dict("records"), nonspeech=NONSPEECH_AFTER),
              per_unseen_system_detection_phone_1pct=per_sys.to_dict(), own_data=None if OWN_REPORT is None else OWN_REPORT.to_dict("records"),
              history=history if "history" in globals() else [], init_checkpoint=str(init_path))
(EXPORT / "posttrain_report.json").write_text(json.dumps(report, indent=1, default=float))
sums = {f: sha256(EXPORT / f) for f in ("detector_state.pt", "detector_config.json")}
(EXPORT / "SHA256SUMS.txt").write_text("".join(f"{h}  {f}\n" for f, h in sums.items()))
print("Saved to", EXPORT); print(json.dumps(sums, indent=1))
print(json.dumps({k: meta[k] for k in ("logit_threshold_eer", "logit_threshold_1pct_fpr", "val_eer")}, indent=1))

## 17. Next steps (on your Mac, in the repo)
1. Download `export_posttrain_v3/` and copy `detector_state.pt` and `detector_config.json` into `backend/AI Deepfake Pipeline/`. Move the old `best.pt` aside: the loader prefers `detector_state.pt`. Keep the old files to roll back.
2. Re-pin the bundle: `python "backend/AI Deepfake Pipeline/model_integrity.py" --pin "backend/AI Deepfake Pipeline"` and review the diff of `model_integrity.json`.
3. Benchmark against the old model. Copy `model teseting/benchmark/configs/development_silero.json` to a new config with `expected_checkpoint_sha256` set to the new hash, point `--model-dir` at a folder containing the new files, and use a new `--output` run name.
4. Decide only if the benchmark and your own recordings improve. Check whether the `gap2s`, `noise10` and non-speech rows got better without new false warnings on human speech. If the kept epoch is 0, the fine-tune didn't help; keep the old model.
5. Deploy: upload `detector_state.pt` as a new GitHub release asset, then update the asset URL, file name and SHA-256 in `deploy/livekit/prepare.py`.

**What this run can't tell you:** LibriSpeech and M-AILABS are both read audiobooks, and every fake is TTS from MLAAD. Real scam calls are conversational and go through real phone networks. Treat these numbers as development results. The strongest evidence for judges is consented real calls through the live system (`RECORD_CALLER_AUDIO=1` with everyone's permission).